In [0]:
%sql
-- ============================================================================
-- Tabela......: mart_carteira_executiva
-- Camada......: Gold (Data Product)
--
-- Objetivo
-- ----------------------------------------------------------------------------
-- Consolidar mensalmente os principais indicadores executivos da carteira
-- de crédito para consumo direto pela página "Executive Overview"
-- do Power BI.
--
-- Perguntas de negócio respondidas
-- ----------------------------------------------------------------------------
-- • A carteira está crescendo?
-- • Estamos originando contratos maiores?
-- • Quanto dinheiro foi contratado?
-- • Quanto esperamos receber?
-- • Quanto efetivamente recebemos?
-- • Estamos recebendo o esperado?
-- • Quanto ainda falta receber?
-- • A inadimplência está aumentando?
--
-- Grão
-- ----------------------------------------------------------------------------
-- 1 linha por competência (dt_competencia)
--
-- Fontes
-- ----------------------------------------------------------------------------
-- gold.fact_carteira_credito
-- gold.fact_parcela_credito
-- ============================================================================

CREATE OR REPLACE TABLE dev_credit_risk.gold.mart_carteira_executiva

USING DELTA

AS

-- ============================================================================
-- Indicadores da Carteira
-- ============================================================================

WITH carteira AS (

SELECT

    date_trunc('MONTH', dt_contratacao) AS dt_competencia,

    COUNT(*)                            AS qtd_contratos,

    SUM(valor_contratado)               AS valor_contratado,

    AVG(valor_contratado)               AS ticket_medio,

    AVG(prazo_meses)                    AS prazo_medio,

    AVG(taxa_juros)                     AS taxa_media_juros


FROM dev_credit_risk.gold.fact_carteira_credito

GROUP BY

    date_trunc('MONTH', dt_contratacao)

),

-- ============================================================================
-- Indicadores de Cobrança
-- ============================================================================

parcelas AS (

SELECT

    dt_competencia,

    SUM(valor_parcela)                  AS valor_esperado,

    SUM(COALESCE(valor_pago,0))         AS valor_recebido,

    SUM(valor_em_aberto)                AS saldo_em_aberto,

    ROUND(

        SUM(COALESCE(valor_pago,0))
        /
        NULLIF(SUM(valor_parcela),0)

    ,4)                                 AS indice_recebimento,

    ROUND(

        SUM(flag_pagamento_atrasado)
        /
        COUNT(*)

    ,4)                                 AS tx_atraso,

    ROUND(

        SUM(flag_pagamento_default_90)
        /
        COUNT(*)

    ,4)                                 AS tx_default,

    COUNT(

        DISTINCT CASE

            WHEN flag_pagamento_default_90 = 1

            THEN id_contrato

        END

    )                                   AS contratos_default

FROM dev_credit_risk.gold.fact_parcela_credito

GROUP BY

    dt_competencia

)

-- ============================================================================
-- Resultado Final
-- ============================================================================

SELECT

    p.dt_competencia,

    --------------------------------------------------------------------------
    -- Carteira
    --------------------------------------------------------------------------

    c.qtd_contratos,

    c.valor_contratado,

    c.ticket_medio,

    c.prazo_medio,

    c.taxa_media_juros,


    --------------------------------------------------------------------------
    -- Cobrança
    --------------------------------------------------------------------------

    p.valor_esperado,

    p.valor_recebido,

    p.indice_recebimento,

    p.saldo_em_aberto,

    --------------------------------------------------------------------------
    -- Qualidade da Carteira
    --------------------------------------------------------------------------

    p.contratos_default,

    p.tx_atraso,

    p.tx_default,

    --------------------------------------------------------------------------
    -- Auditoria
    --------------------------------------------------------------------------

    current_timestamp() AS dt_carga

FROM parcelas p

LEFT JOIN carteira c

ON p.dt_competencia = c.dt_competencia

ORDER BY

    p.dt_competencia;

In [0]:
%sql
select * from dev_credit_risk.gold.mart_carteira_executiva